# Check that the data is stored as expected.

## Import Libraries.

In [1]:
import os

import anndata as ad
import pandas as pd

### Constants for loops to run.

In [2]:
RUN_LOOP1 = False
RUN_LOOP2 = False
RUN_LOOP3 = True

## Read Metadata.

### Read metadata csv.

In [3]:
metadata_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/new_dataset/metadata_updated.xlsx"
meta_df = pd.read_excel(metadata_path)
meta_df

,experiment_id,740-yp_[µm],mtg_[µm],rhflt3l_[ng_ml],experiment_number,gm-csf_[ng_ml],tpo_[ng_ml],sr1_[nm],um171_[nm],um729_[µm],...,source_id,hydrogel_type,o2_[%],source_cell_phenotype,total_count_beads,counts_source_cell_phenotype,days_of_culture,plate,absolute_number_multiplier_[2^n],cytometer_id
0,LPHO01,5,100,10,1.0,0,0.0,0,0,1.5,...,pooled,cellbind,20,CD34,1994.0,10000,14,2.0,0,sonic
1,LPHO01,5,100,10,2.0,0,0.0,75,0,1.5,...,pooled,cellbind,20,CD34,1994.0,10000,14,2.0,0,sonic
2,LPHO01,5,100,10,3.0,0,0.0,375,0,1.5,...,pooled,cellbind,20,CD34,1994.0,10000,14,2.0,0,sonic
3,LPHO01,5,100,10,4.0,0,0.0,0,0,1.5,...,pooled,cellbind,20,CD34,1994.0,10000,14,2.0,2,sonic
4,LPHO01,5,100,10,5.0,0,0.0,0,0,1.5,...,pooled,cellbind,20,CD34,1994.0,10000,14,2.0,2,sonic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,JAR03,5,100,10,211.0,10,2.5,750,560,0.0,...,cb1,cellbind,20,CD34,1994.0,10000,13,1.0,1,sonic
269,JAR03,5,100,10,212.0,10,2.5,750,560,0.0,...,cb2,cellbind,20,CD34,1994.0,10000,13,1.0,1,sonic
270,JAR02,5,100,10,213.0,10,2.5,750,1120,0.0,...,pooled,cellbind,20,CD34,1994.0,10000,13,2.0,0,sonic
271,JAR02,5,100,10,214.0,10,2.5,750,0,1.5,...,pooled,cellbind,20,CD34,1994.0,10000,13,2.0,0,sonic


### Split data between loop 0 and loop1.

In [4]:
# ---- Filter metadata ----
# we want to exclude all the LPHO012 experiments
# but the number 205, which was already seen by the model 
meta_lpho012_mask = meta_df.experiment_id == "LPHO012"
meta_is_205_mask = meta_df.experiment_number == 205
meta_loop1_mask = meta_lpho012_mask & ~meta_is_205_mask
meta_loop0_mask = ~meta_loop1_mask

# display number of retained conditions
meta_n_loop0 = meta_loop0_mask.sum()
meta_n_loop1 = meta_loop1_mask.sum()
print(f"{meta_n_loop0=}, {meta_n_loop1=}")

# filter data frames
meta_df_looop0 = meta_df[meta_loop0_mask]
meta_df_looop1 = meta_df[meta_loop1_mask]

# get loop1 experiment numbers
loop0_exp_numbers = meta_df_looop0.experiment_number.to_list()
loop1_exp_numbers = meta_df_looop1.experiment_number.to_list()
print(f"{loop1_exp_numbers=}")


meta_n_loop0=np.int64(268), meta_n_loop1=np.int64(5)
loop1_exp_numbers=[206.0, 207.0, 208.0, 209.0, 210.0]


## Loop 1.

### Load input data.

In [5]:
if RUN_LOOP1:
    # ---- Load 500k subset ----
    adata_loop1_concat_500k_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/new_dataset/BloodPlus_celltype_pm_500k_.h5ad"
    adata_loop1_concat_500k = ad.read_h5ad(adata_loop1_concat_500k_path)
    print(f"{adata_loop1_concat_500k=}")

    # ---- Load full data ----
    adata_loop1_concat_full_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/new_dataset/BloodPlus_Logicle_pm_.h5ad"
    adata_loop1_concat_full = ad.read_h5ad(adata_loop1_concat_full_path)
    print(f"{adata_loop1_concat_full=}")

### Create masks for anndata.

In [6]:
if RUN_LOOP1:
    # ---- Create mask for 500k subset ---
    loop1_500k_concat_mask = adata_loop1_concat_500k.obs.experiment_number.map(lambda e: e in loop1_exp_numbers)
    loop0_500k_concat_mask = ~loop1_500k_concat_mask
    n_cells_loop0_concat_500k = loop0_500k_concat_mask.sum()
    n_cells_loop1_concat_500k = loop1_500k_concat_mask.sum()
    print(f"{n_cells_loop0_concat_500k=}, {n_cells_loop1_concat_500k=}")

    # ---- Create mask for 500k subset ---
    loop1_concat_full_mask = adata_loop1_concat_full.obs.experiment_number.map(lambda e: e in loop1_exp_numbers)
    loop0_concat_full_mask = ~loop1_concat_full_mask
    n_cells_loop0_concat_full = loop0_concat_full_mask.sum()
    n_cells_loop1_concat_full = loop1_concat_full_mask.sum()
    print(f"{n_cells_loop0_concat_full=}, {n_cells_loop1_concat_full=}")


### Slice the adata.

In [7]:
if RUN_LOOP1:
    # ---- Slice 500k subset ----
    adata_loop0_500k = adata_loop1_concat_500k[loop0_500k_concat_mask]
    adata_loop1_500k = adata_loop1_concat_500k[loop1_500k_concat_mask]
    print(f"{adata_loop0_500k=}\n{adata_loop1_500k=}")

    # ---- Slice full data ----
    adata_loop0_full = adata_loop1_concat_full[loop0_concat_full_mask]
    adata_loop1_full = adata_loop1_concat_full[loop1_concat_full_mask]
    print(f"{adata_loop0_full=}\n{adata_loop1_full=}")

### Sanity check for loop 0 data.

In [8]:
if RUN_LOOP1:
    # ---- Check that loop 0 500k subset does not contain experiment numbers beyond 205 ----
    loop0_500k_g205 = adata_loop0_500k.obs.experiment_number.map(lambda e: e%1000) > 206 # module 1000 because of numbering strategy
    n_loop0_500k_g205 = loop0_500k_g205.sum()
    assert n_loop0_500k_g205 == 0, n_loop0_500k_g205

    # ---- Check that loop 0 full data does not contain experiment numbers beyond 205 ----
    loop0_full_g205 = adata_loop0_full.obs.experiment_number.map(lambda e: e%1000) > 206 # module 1000 because of numbering strategy
    n_loop0_full_g205 = loop0_full_g205.sum()
    assert n_loop0_full_g205 == 0, n_loop0_full_g205


### Sanity check for loop 1 data.

In [9]:
if RUN_LOOP1:
    # ---- Check that loop 1 500k subset contains only expected experiment numbers ----
    loop1_500k_wexp = adata_loop1_500k.obs.experiment_number.map(lambda e: e not in loop1_exp_numbers)
    n_loop1_500k_wexp = loop1_500k_wexp.sum()
    assert n_loop1_500k_wexp == 0, n_loop1_500k_wexp

    # ---- Check that loop 1 full data contains only expected experiment numbers ----
    loop1_full_wexp = adata_loop1_full.obs.experiment_number.map(lambda e: e not in loop1_exp_numbers)
    n_loop1_full_wexp = loop1_full_wexp.sum()
    assert n_loop1_full_wexp == 0, n_loop1_full_wexp


### Write data loop 0.

In [ ]:
# ---- Define output paths ----
loop0_dump_dir = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop0"
loop0_500k_out_path = os.path.join(loop0_dump_dir, "adata_500k.h5ad")
loop0_full_out_path = os.path.join(loop0_dump_dir, "adata_full.h5ad")

# ---- Write loop 0 data ----
if RUN_LOOP1:
    adata_loop0_500k.write_h5ad(loop0_500k_out_path)
    adata_loop0_full.write_h5ad(loop0_full_out_path)
# ---- Read loop 0 data ----
else:
    adata_loop0_500k = ad.read_h5ad(loop0_500k_out_path)
    adata_loop0_full = ad.read_h5ad(loop0_full_out_path)


### Write adata for loop 1.

In [ ]:
# ---- Define output paths ----
loop1_dump_dir = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop1"
loop1_500k_res_out_path = os.path.join(loop1_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop1_full_res_out_path = os.path.join(loop1_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop1_500k_out_path = os.path.join(loop1_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop1_full_out_path = os.path.join(loop1_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Write loop 1 data ----
if RUN_LOOP1:
    adata_loop1_500k.write_h5ad(loop1_500k_res_out_path)
    adata_loop1_full.write_h5ad(loop1_full_res_out_path)
    adata_loop1_concat_500k.write_h5ad(loop1_500k_out_path)
    adata_loop1_concat_full.write_h5ad(loop1_full_out_path)
# ---- Read loop 1 data ----
else:
    adata_loop1_500k = ad.read_h5ad(loop1_500k_res_out_path)
    adata_loop1_full = ad.read_h5ad(loop1_full_res_out_path)
    adata_loop1_concat_500k = ad.read_h5ad(loop1_500k_out_path)
    adata_loop1_concat_full = ad.read_h5ad(loop1_full_out_path)

## Loop 2.

### Load input data.

In [ ]:
if RUN_LOOP2:
    # ---- Load 500k subset ----
    adata_loop2_500k_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/loop2/BloodPlus_Loop2_logicle_500k.h5ad"
    adata_loop2_500k = ad.read_h5ad(adata_loop2_500k_path)
    print(f"{adata_loop2_500k=}")

    # ---- Load 500k subset ----
    adata_loop2_full_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/loop2/BloodPlus_Loop2_logicle.h5ad"
    adata_loop2_full = ad.read_h5ad(adata_loop2_full_path)
    print(f"{adata_loop2_full=}")


### Get loop 2 experiment numbers.

In [ ]:
if RUN_LOOP2:
    loop2_exp_numbers = adata_loop2_500k.obs.experiment_number.unique().to_list()
    loop2_exp_numbers = sorted(loop2_exp_numbers)
    print(f"{loop2_exp_numbers=}")

### Concatenate with data from loop 1.

In [ ]:
if RUN_LOOP2:
    # ---- Concatenate 500k subset ----
    adata_loop2_concat_500k = ad.concat((adata_loop1_concat_500k, adata_loop2_500k), merge="same", uns_merge="same")
    adata_loop2_concat_500k.obs_names_make_unique()
    print(f"{adata_loop2_concat_500k=}")

    # ---- Concatenate full data ----
    adata_loop2_concat_full = ad.concat((adata_loop1_concat_full, adata_loop2_full), merge="same", uns_merge="same")
    adata_loop2_concat_full.obs_names_make_unique()
    print(f"{adata_loop2_concat_full=}")


### Handle column type for saving.

In [ ]:
# ---- Define function for sanitizing adata ----
def sanitize_anndata(adata: ad.AnnData, convert_to: str = "string") -> ad.AnnData:
    for df, name in [(adata.obs, "obs"), (adata.var, "var")]:
        for col in df.columns:
            if df[col].dtype == "object":
                if convert_to == "string":
                    # Convert to pandas nullable string dtype (safe for HDF5 writing)
                    df[col] = df[col].astype("string")
                elif convert_to == "category":
                    df[col] = df[col].astype("category")
                else:
                    raise ValueError("convert_to must be 'string' or 'category'")
    return adata

if RUN_LOOP2:
    # ---- Sanitizing concatenated data ----
    adata_loop2_concat_full = sanitize_anndata(adata_loop2_concat_full)
    adata_loop2_concat_500k = sanitize_anndata(adata_loop2_concat_500k)

### Write data from loop2.

In [ ]:
# ---- Define output paths ----
loop2_dump_dir = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/loops/data/loop2"
loop2_500k_res_out_path = os.path.join(loop2_dump_dir, "adata_500k_residual.h5ad") # residual adata (500k)
loop2_full_res_out_path = os.path.join(loop2_dump_dir, "adata_full_residual.h5ad") # residual adata (full)
loop2_500k_out_path = os.path.join(loop2_dump_dir, "adata_500k.h5ad") # concatenated adata (500k)
loop2_full_out_path = os.path.join(loop2_dump_dir, "adata_full.h5ad") # concatenate adata (full)

# ---- Write 500k subset ----
if RUN_LOOP2:
    adata_loop2_500k.write_h5ad(loop2_500k_res_out_path)
    adata_loop2_full.write_h5ad(loop2_full_res_out_path)
    adata_loop2_concat_500k.write_h5ad(loop2_500k_out_path)
    adata_loop2_concat_full.write_h5ad(loop2_full_out_path)
# ---- Read 500k subset ----
else:
    adata_loop2_500k = ad.read_h5ad(loop2_500k_res_out_path)
    adata_loop2_full = ad.read_h5ad(loop2_full_res_out_path)
    adata_loop2_concat_500k = ad.read_h5ad(loop2_500k_out_path)
    adata_loop2_concat_full = ad.read_h5ad(loop2_full_out_path)

## Loop2: Prepare splits.

### Define mapping to experiment type.

In [17]:
# ---- Define mapping from experiment number to run type ----
settings_experiments_map_loop2 = {
    216: 'late_mgkPro_low_u',
    217: 'late_mgkPro_high_u0',
    218: 'late_mgkPro_high_u1',
    219: 'EryPro_low_u',
    220: 'EryPro_high_u0',
    221: 'EryPro_high_u1',
    222: 'HSC_low_u',
    223: 'HSC_high_u0',
    224: 'HSC_high_u1',
    225: 'preProB_low_u',
    226: 'preProB_high_u0',
    227: 'preProB_high_u1',
}

# ---- Group the uncertain experiments for train and validation ----
loop2_group0_exp = [
    k for k, v in settings_experiments_map_loop2.items()
    if v.endswith("0")
]
loop2_group1_exp = [
    k for k, v in settings_experiments_map_loop2.items()
    if v.endswith("1")
]
print(f"{loop2_group0_exp=}, {loop2_group1_exp=}")

loop2_group0_exp=[217, 220, 223, 226], loop2_group1_exp=[218, 221, 224, 227]


### Subset the loop2 data based on the group.

In [18]:
if RUN_LOOP2:
    # --- Subset for group 1 ----
    adata_loop2_500k_group0_mask = adata_loop2_500k.obs.experiment_number.astype(int).map(lambda e: e in loop2_group0_exp)
    adata_loop2_500k_group1_mask = ~adata_loop2_500k_group0_mask
    adata_loop2_500k_group1 = adata_loop2_500k[adata_loop2_500k_group1_mask]
    print(f"{adata_loop2_500k_group1=}")

    adata_loop2_concat_500k_group0_mask = adata_loop2_concat_500k.obs.experiment_number.astype(int).map(lambda e: e in loop2_group0_exp)
    adata_loop2_concat_500k_group1_mask = ~adata_loop2_concat_500k_group0_mask
    adata_loop2_concat_500k_group1 = adata_loop2_concat_500k[adata_loop2_concat_500k_group1_mask]
    print(f"{adata_loop2_concat_500k_group1=}")

    adata_loop2_full_group0_mask = adata_loop2_full.obs.experiment_number.astype(int).map(lambda e: e in loop2_group0_exp)
    adata_loop2_full_group1_mask = ~adata_loop2_full_group0_mask
    adata_loop2_full_group1 = adata_loop2_full[adata_loop2_full_group1_mask]
    print(f"{adata_loop2_full_group1=}")

    adata_loop2_concat_full_group0_mask = adata_loop2_concat_full.obs.experiment_number.astype(int).map(lambda e: e in loop2_group0_exp)
    adata_loop2_concat_full_group1_mask = ~adata_loop2_concat_full_group0_mask
    adata_loop2_concat_full_group1 = adata_loop2_concat_full[adata_loop2_concat_full_group1_mask]
    print(f"{adata_loop2_concat_full_group1=}")

    # --- Subset for group 0 ----
    adata_loop2_500k_group1_mask = adata_loop2_500k.obs.experiment_number.astype(int).map(lambda e: e in loop2_group1_exp)
    adata_loop2_500k_group0_mask = ~adata_loop2_500k_group1_mask
    adata_loop2_500k_group0 = adata_loop2_500k[adata_loop2_500k_group0_mask]
    print(f"{adata_loop2_500k_group0=}")

    adata_loop2_concat_500k_group1_mask = adata_loop2_concat_500k.obs.experiment_number.astype(int).map(lambda e: e in loop2_group1_exp)
    adata_loop2_concat_500k_group0_mask = ~adata_loop2_concat_500k_group1_mask
    adata_loop2_concat_500k_group0 = adata_loop2_concat_500k[adata_loop2_concat_500k_group0_mask]
    print(f"{adata_loop2_concat_500k_group0=}")

    adata_loop2_full_group1_mask = adata_loop2_full.obs.experiment_number.astype(int).map(lambda e: e in loop2_group1_exp)
    adata_loop2_full_group0_mask = ~adata_loop2_full_group0_mask
    adata_loop2_full_group0 = adata_loop2_full[adata_loop2_full_group0_mask]
    print(f"{adata_loop2_full_group0=}")

    adata_loop2_concat_full_group1_mask = adata_loop2_concat_full.obs.experiment_number.astype(int).map(lambda e: e in loop2_group1_exp)
    adata_loop2_concat_full_group0_mask = ~adata_loop2_concat_full_group1_mask
    adata_loop2_concat_full_group0 = adata_loop2_concat_full[adata_loop2_concat_full_group0_mask]
    print(f"{adata_loop2_concat_full_group0=}")


### Split the data between groups 0 and 1.

In [19]:
if RUN_LOOP2:
    # ---- Define dictionary for group 0 ----
    loop2_adata_dictionary_group0 = {
        "adata_500k_residual": adata_loop2_500k_group0,
        "adata_residual": adata_loop2_full_group0,
        "adata_500k": adata_loop2_concat_500k_group0,
        "adata_full": adata_loop2_concat_full_group0,
    }

    # ---- Define dictionary for group 1 ----
    loop2_adata_dictionary_group1 = {
        "adata_500k_residual": adata_loop2_500k_group1,
        "adata_residual": adata_loop2_full_group1,
        "adata_500k": adata_loop2_concat_500k_group1,
        "adata_full": adata_loop2_concat_full_group1,
    }

    # ---- Define per loop dictionary ----
    loop2_groups_adatas_dicts = {
        "0": loop2_adata_dictionary_group0,
        "1": loop2_adata_dictionary_group1
    }

    # ---- Iterate over dictionaries ----
    for group_id, adata_group_dict in loop2_groups_adatas_dicts.items():
        # ---- Define paths ----
        group_dir = os.path.join(loop2_dump_dir, "splits", group_id)
        adata_res_500k_path = os.path.join(group_dir, "adata_500k_residual.h5ad")
        adata_res_full_path = os.path.join(group_dir, "adata_full_residual.h5ad")
        adata_500k_path = os.path.join(group_dir, "adata_500k.h5ad")
        adata_full_path = os.path.join(group_dir, "adata_full.h5ad")

        # ---- Extract data from dictionaries ----
        adata_500k_res = sanitize_anndata(adata_group_dict["adata_500k_residual"])
        adata_full_res = sanitize_anndata(adata_group_dict["adata_residual"])
        adata_500k = sanitize_anndata(adata_group_dict["adata_500k"])
        adata_full = sanitize_anndata(adata_group_dict["adata_full"])

        # ---- Write data to disk ----
        adata_500k_res.write_h5ad(adata_res_500k_path)
        adata_full_res.write_h5ad(adata_res_full_path)
        adata_500k.write_h5ad(adata_500k_path)
        adata_full.write_h5ad(adata_full_path)


## Loop 3.

### Load input data.

In [20]:
if RUN_LOOP3:
    # ---- Load 500k subset ----
    adata_loop3_500k_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/loop3/BloodPlus_Loop3_logicle_500k.h5ad"
    adata_loop3_500k = ad.read_h5ad(adata_loop3_500k_path)
    print(f"{adata_loop3_500k=}")

    # ---- Load 500k subset ----
    adata_loop3_full_path = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/loop3/BloodPlus_Loop3_logicle.h5ad"
    adata_loop3_full = ad.read_h5ad(adata_loop3_full_path)
    print(f"{adata_loop3_full=}")


adata_loop3_500k=AnnData object with n_obs × n_vars = 499994 × 20
    obs: 'experiment_number', 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'il7_[ng_ml]', 'mcsf_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'TIME', 'NA', 'AF Index', 'batch', '7-AAD'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnR', 'PnD', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps'
    layers: 'positives', 'raw'
adata_loop3_

### Get experiment numbers from loop 3.

In [21]:
if RUN_LOOP3:
    loop3_exp_numbers = adata_loop3_500k.obs.experiment_number.unique().to_list()
    loop3_exp_numbers = sorted(loop3_exp_numbers)
    print(f"{loop3_exp_numbers=}")

loop3_exp_numbers=['228', '229', '230', '231', '232', '233', '234', '235', '236', '237', '238']


/tmp/ipykernel_1732126/712369097.py:2: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  loop3_exp_numbers = adata_loop3_500k.obs.experiment_number.unique().to_list()
